# Kiến trúc 3 Models — Sơ đồ khối

Vẽ block diagram cho Baseline CNN, DAN, POSTER — dùng làm mẫu vẽ tay nộp thầy.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
import numpy as np

plt.rcParams['figure.dpi'] = 150
plt.rcParams['font.size'] = 11

# Color palette
C = {
    'input':    '#E8F5E9',  # xanh lá nhạt
    'conv':     '#BBDEFB',  # xanh dương nhạt
    'pool':     '#FFF9C4',  # vàng nhạt
    'fc':       '#FFE0B2',  # cam nhạt
    'output':   '#FFCDD2',  # đỏ nhạt
    'attention':'#E1BEE7',  # tím nhạt
    'fusion':   '#B2EBF2',  # cyan nhạt
    'backbone': '#C8E6C9',  # xanh lá
    'text':     '#2C3E50',
    'arrow':    '#546E7A',
    'box_edge': '#37474F',
}

In [ ]:
def draw_block(ax, x, y, w, h, label, sublabel='', color='#BBDEFB', edgecolor='#37474F'):
    """Vẽ 1 block chữ nhật bo góc."""
    box = FancyBboxPatch((x, y), w, h,
                         boxstyle="round,pad=0.1",
                         facecolor=color, edgecolor=edgecolor,
                         linewidth=1.5, zorder=2)
    ax.add_patch(box)
    ax.text(x + w/2, y + h/2, label,
            ha='center', va='center', fontsize=9, fontweight='bold',
            color=C['text'], zorder=3)
    if sublabel:
        ax.text(x + w/2, y + h*0.25, sublabel,
                ha='center', va='center', fontsize=7,
                color='#546E7A', zorder=3)


def draw_arrow(ax, x1, y1, x2, y2, label=''):
    """Vẽ mũi tên giữa 2 block."""
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color=C['arrow'],
                               lw=1.5, connectionstyle='arc3,rad=0'),
                zorder=1)
    if label:
        mx, my = (x1 + x2) / 2, (y1 + y2) / 2
        ax.text(mx, my + 0.3, label, ha='center', va='bottom',
                fontsize=7, color='#546E7A', style='italic')


def draw_baseline_cnn(ax):
    ax.set_title('Baseline CNN', fontsize=14, fontweight='bold', pad=15)
    BX, BY = 0.5, 0
    BW, BH = 1.8, 0.9

    blocks = [
        (BX, BY+7.2, BW, BH, 'Input\n100×100×3', '', C['input']),
        (BX, BY+5.9, BW, BH, 'ConvBlock 1\n64 filters ×2', '3×3 conv + BN + ReLU\nMaxPool 3×3', C['conv']),
        (BX, BY+4.6, BW, BH, 'ConvBlock 2\n96 filters ×3', '3×3 conv + BN + ReLU\nMaxPool 3×3', C['conv']),
        (BX, BY+3.3, BW, BH, 'ConvBlock 3\n128 filters ×3', '3×3 conv + BN + ReLU', C['conv']),
        (BX, BY+2.0, BW, 0.8, 'GAP', 'Global Avg Pooling', C['pool']),
        (BX, BY+1.0, BW, 0.7, 'Dropout', 'rate=0.25', C['pool']),
        (BX, BY-0.2, BW, BH, 'Dense + Softmax\n7 classes', '', C['output']),
    ]

    for x, y, w, h, label, sub, color in blocks:
        draw_block(ax, x, y, w, h, label, sub, color)

    for i in range(len(blocks) - 1):
        _, y1, _, h1, _, _, _ = blocks[i]
        _, y2, _, _, _, _, _ = blocks[i+1]
        draw_arrow(ax, BX + BW/2, y1 - 0.05, BX + BW/2, y2 + blocks[i+1][3])

    ax.set_xlim(0, 3)
    ax.set_ylim(-0.5, 8.5)
    ax.axis('off')

    # Parameter annotation
    ax.text(BX + BW/2, -0.5, 'Params: ~670K', ha='center', fontsize=9,
            style='italic', color='#546E7A')

In [ ]:
def draw_dan(ax):
    ax.set_title('DAN — Distract Your Attention Network', fontsize=14, fontweight='bold', pad=15)
    CX, CY = 0.5, 0
    W2 = 2.0

    blocks_dan = [
        (CX, CY+7.8, W2, 1.0, 'Input\n224×224×3', '', C['input']),
        (CX, CY+6.2, W2, 1.3, 'ResNet18 Backbone', '(trừ FC layer)\n→ feature map 512×7×7', C['backbone']),
        (CX, CY+4.6, W2, 1.0, 'Conv2D 1×1\n512 → 4 heads', 'conv_att', C['attention']),
        (CX, CY+3.2, W2, 1.0, 'Softmax Attention\n4 heads × 7×7', 'mỗi head học 1 vùng', C['attention']),
        (CX, CY+1.8, W2, 1.0, 'Weighted Sum\n+ Mean fusion', 'gộp 4 heads → 512-d', C['fusion']),
        (CX, CY-0.3, W2, 1.3, 'FC + BatchNorm\n→ 7 emotions', 'softmax', C['output']),
    ]

    for x, y, w, h, label, sub, color in blocks_dan:
        draw_block(ax, x, y, w, h, label, sub, color)

    for i in range(len(blocks_dan) - 1):
        _, y1, _, h1, _, _, _ = blocks_dan[i]
        _, y2, _, _, _, _, _ = blocks_dan[i+1]
        draw_arrow(ax, CX + W2/2, y1 - 0.05, CX + W2/2, y2 + blocks_dan[i+1][3])

    # Attention heads illustration
    head_colors = ['#EF9A9A', '#CE93D8', '#81D4FA', '#A5D6A7']
    head_labels = ['Head 1', 'Head 2', 'Head 3', 'Head 4']
    for i in range(4):
        hx = 3.5 + i * 0.7
        hy = 4.0
        circle = plt.Circle((hx, hy), 0.25, color=head_colors[i], ec='#37474F', lw=1, zorder=2)
        ax.add_patch(circle)
        ax.text(hx, hy, str(i+1), ha='center', va='center', fontsize=8, fontweight='bold')
        ax.text(hx, hy - 0.45, head_labels[i], ha='center', fontsize=6, color='#546E7A')

    # Arrow from conv_att to heads
    for i in range(4):
        hx = 3.5 + i * 0.7
        draw_arrow(ax, CX + W2, 4.6 + 0.5, hx, 4.25)

    # Arrow from heads to weighted sum
    for i in range(4):
        hx = 3.5 + i * 0.7
        draw_arrow(ax, hx, 3.75, CX + W2/2, 2.8)

    ax.set_xlim(0, 6.5)
    ax.set_ylim(-0.5, 9.5)
    ax.axis('off')

    ax.text(CX + W2/2, -0.5, 'Params: ~11M', ha='center', fontsize=9,
            style='italic', color='#546E7A')

In [ ]:
def draw_poster(ax):
    ax.set_title('POSTER — Pyramid Fusion Transformer', fontsize=14, fontweight='bold', pad=15)

    # ─── IR-50 branch (trái) ───
    ix, iy = 0.3, 1.0
    iw = 2.0

    blocks_ir = [
        (ix, iy+7.0, iw, 1.0, 'Input\n224×224×3', '', C['input']),
        (ix, iy+5.5, iw, 1.2, 'IR-50 Backbone\n(Improved ResNet-50)', 'SE blocks\n→ 512×7×7', C['backbone']),
        (ix, iy+4.0, iw, 1.0, 'Reshape\n512×7×7 → 49×1024', '', C['conv']),
        (ix, iy+2.5, iw, 1.0, 'Linear\n1024 → 512', 'ir_layer', C['conv']),
    ]

    for x, y, w, h, label, sub, color in blocks_ir:
        draw_block(ax, x, y, w, h, label, sub, color)

    for i in range(len(blocks_ir) - 1):
        _, y1, _, h1, _, _, _ = blocks_ir[i]
        _, y2, _, _, _, _, _ = blocks_ir[i+1]
        draw_arrow(ax, ix + iw/2, y1 - 0.05, ix + iw/2, y2 + blocks_ir[i+1][3])

    # ─── MobileFaceNet branch (phải) ───
    mx, my = 4.0, 1.0
    mw = 2.0

    blocks_mfn = [
        (mx, my+7.0, mw, 1.0, 'Input (resized)\n112×112×3', 'F.interpolate', C['input']),
        (mx, my+5.5, mw, 1.2, 'MobileFaceNet\n(landmark)', 'depthwise conv\n→ 136-d features', C['backbone']),
        (mx, my+4.0, mw, 1.0, 'Reshape + Transpose\n→ 49×512', '', C['conv']),
    ]

    for x, y, w, h, label, sub, color in blocks_mfn:
        draw_block(ax, x, y, w, h, label, sub, color)

    for i in range(len(blocks_mfn) - 1):
        _, y1, _, h1, _, _, _ = blocks_mfn[i]
        _, y2, _, _, _, _, _ = blocks_mfn[i+1]
        draw_arrow(ax, mx + mw/2, y1 - 0.05, mx + mw/2, y2 + blocks_mfn[i+1][3])

    # ─── Fusion & Head ───
    fx, fy = 2.3, 0
    fw = 2.0

    # Arrow từ 2 nhánh vào fusion
    draw_arrow(ax, ix + iw, 3.8, fx + fw/2, fy + 6.7)
    draw_arrow(ax, mx, 3.8, fx + fw/2, fy + 6.7)
    ax.text(fx + fw/2, fy + 6.9, 'concat', ha='center', fontsize=8,
            style='italic', color='#546E7A')

    draw_block(ax, fx, fy+5.0, fw, 1.5, 'HyVisionTransformer\n(Pyramid Fusion)', 'cross-attention 8 heads\n8 layers × 3 scales', C['fusion'])
    draw_arrow(ax, fx + fw/2, fy+5.0 - 0.05, fx + fw/2, fy+3.0 + 1.2)

    draw_block(ax, fx, fy+3.0, fw, 1.2, 'SE Block\n(Channel Attention)', 'squeeze-excitation', C['attention'])
    draw_arrow(ax, fx + fw/2, fy+3.0 - 0.05, fx + fw/2, fy+1.5 + 0.8)

    draw_block(ax, fx, fy+1.5, fw, 0.8, 'Dropout', 'rate=0.3', C['pool'])
    draw_arrow(ax, fx + fw/2, fy+1.5 - 0.05, fx + fw/2, fy+0.0 + 1.0)

    draw_block(ax, fx, fy+0.0, fw, 1.0, 'Linear Head\n512 → 7 emotions', 'softmax', C['output'])

    ax.set_xlim(0, 6.5)
    ax.set_ylim(-0.5, 9.0)
    ax.axis('off')

    ax.text(fx + fw/2, -0.5, 'Params: ~58M', ha='center', fontsize=9,
            style='italic', color='#546E7A')

In [ ]:
# ─── Vẽ 3 diagrams ───
fig, axes = plt.subplots(1, 3, figsize=(24, 10))

draw_baseline_cnn(axes[0])
draw_dan(axes[1])
draw_poster(axes[2])

plt.suptitle('Facial Emotion Recognition — 3 Model Architectures',
             fontsize=16, fontweight='bold', y=0.98)
plt.tight_layout()
plt.savefig('architecture_diagrams.png', dpi=200, bbox_inches='tight', facecolor='white')
plt.show()
print('Saved: architecture_diagrams.png')

---
## Hướng dẫn vẽ tay

### Baseline CNN
- 3 ConvBlock chồng lên nhau, filters tăng dần (64→96→128)
- Mỗi block: Conv 3×3 + BN + ReLU (lặp 2-3 lần) → MaxPool 3×3
- Cuối: GAP → Dropout → Dense 7 (softmax)

### DAN
- ResNet18 backbone trích feature map 512×7×7
- Conv 1×1 → 4 attention heads, mỗi head softmax riêng
- Weighted sum từng head → mean fusion → FC + BN → 7 classes

### POSTER
- **2 nhánh song song**: IR-50 (ảnh gốc) + MobileFaceNet (landmark)
- IR-50: 224→112→7×7 feature → reshape 49×1024 → Linear 512
- MFN: 112→112→136-d landmark → reshape 49×512
- **Fusion**: HyViT (cross-attention pyramid) → SE Block → Dropout → Head
- Params ~58M (lớn nhất, chạy Colab)